Cross Entropy Loss Block

In [2]:
import numpy as np

In [3]:
softmax_outputs = np.array([
    [0.7,0.1,0.2],
    [0.1,0.5,0.4],
    [0.02,0.9,0.08]
])

class_targets = [0,1,1]

print(softmax_outputs[[0,1,2], class_targets])

[0.7 0.5 0.9]


In [5]:
print(-np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
]))

neg_log = -np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
])

average_loss = np.mean(neg_log)
print(average_loss)

[0.35667494 0.69314718 0.10536052]
0.38506088005216804


<h3>One hot encoding</h3>

In [10]:
y_true_check = np.array([
    [1,0,0],
    [0,1,0],
    [0,1,0]
])

y_pred_clipped_check = np.array([
    [0.7,0.1,0.2],
    [0.1,0.5,0.4],
    [0.02,0.9,0.08]
])

ans = y_true_check * y_pred_clipped_check
ans_sum = np.sum(ans, axis=1)
neg_log = - np.log(ans_sum)

print(neg_log)
print(np.mean(neg_log))


[0.35667494 0.69314718 0.10536052]
0.38506088005216804


Implementing loss class

In [12]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss

In [13]:
# common loss class
class Loss_CategoricalCrossEntropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples), y_true
            ]
        
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true, axis=1
            )
        
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods 

In [15]:
class_targets = np.array([
    [1,0,0],
    [0,1,0],
    [0,1,0]
])

softmax_outputs = np.array([
    [0.7,0.1,0.2],
    [0.1,0.5,0.4],
    [0.02,0.9,0.08]
])

loss_function = Loss_CategoricalCrossEntropy()
loss = loss_function.calculate(softmax_outputs,class_targets)
print(loss)

0.38506088005216804


<h1>For Spiral Data</h1>

In [16]:
from nnfs.datasets import spiral_data

In [17]:
class DenseLayer:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

In [19]:
# ReLU activation class
class ReLUActivation:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

In [18]:
class SoftmaxActivation:
    def forward(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probabilities

In [22]:
X, y = spiral_data(samples=100, classes=3)

dense1 = DenseLayer(2, 3)
activation1 = ReLUActivation()

dense2 = DenseLayer(3, 3)
activation2 = SoftmaxActivation()

loss_function = Loss_CategoricalCrossEntropy()

dense1.forward(X)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

print(activation2.output[:5])

loss = loss_function.calculate(activation2.output, y)

print("loss", loss)

predictions = np.argmax(activation2.output, axis=1)
if len(y.shape) == 2:
    y = np.argmax(y, axis=1)
accuracy = np.mean(predictions == y)

print("accuracy:", accuracy)

[[0.33333333 0.33333333 0.33333333]
 [0.33333336 0.33333325 0.3333334 ]
 [0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]]
loss 1.0986070592756172
accuracy: 0.33666666666666667


<h1>Accuracy</h1>

In [26]:
softmax_outputs = np.array([
    [0.7,0.1,0.2],
    [0.1,0.5,0.4],
    [0.02,0.9,0.08]
])

class_targets = np.array([0,1,1])

predictions = np.argmax(softmax_outputs, axis=1)

if len(class_targets.shape) == 2:
    class_targets = np.argmax(class_targets, axis=1)
accuracy = np.mean(predictions == class_targets)

print("accuracy", accuracy)

accuracy 1.0
